# Day 2: Diversification opportunities and the product space

ECU Summer School of Economic Complexity. UM6P Rabat. Afternoon Session 2.

Yesterday we measured what a country makes. Today we ask what that implies about where it can go next. The morning lectures gave you the ideas: relatedness as capability overlap, proximity from co-export, the product space, and the principle that economies move into activities related to what they already do. This afternoon each of those becomes a number and a picture for your own country.

Today's skeleton, reused from here to Day 3:

> Pair → Place → Score → Rank

Pair products by how often they are co-exported (proximity). Place your country on the product space. Score every product by how close it is to your current basket (density). Rank the unexploited products into a shortlist worth arguing about. Day 1's chain (Normalize → Benchmark → Binarize → Count) is not gone: section 1 reruns it to rebuild Mcp, and today's steps take over from there.

As always, `COUNTRY`, `COMPARATORS`, and `YEAR` at the top drive every figure. If you are on Google Colab, save your own copy first: File menu, "Save a copy in Drive".

## 0. Setup: load the data packet

Same packet as Day 1, plus two small layout files for the product space. The notebook finds the packet on its own: the course repository's copy if you cloned it, otherwise a one-time download from the course GitHub release.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr

# --- The ONE place you edit to make this notebook about your own country. ------
# Codes are ISO 3166-1 alpha-2 (two letters). MA = Morocco, EG = Egypt,
# TN = Tunisia, KR = Republic of Korea, DE = Germany. See reference/units.csv.
COUNTRY = "MA"  # the focus country (Morocco)
COMPARATORS = ["EG", "TN", "KR"]  # Egypt, Tunisia (MENA) + Korea (frontier)
YEAR = 2023  # latest year in the packet
# -------------------------------------------------------------------------------

LOCAL_PACKET = Path("../../data/processed/morocco_data_packet")
DATA_RELEASE = "https://github.com/shreyasgm/ecu-complexity-labs/releases/download/data-v1"

if LOCAL_PACKET.exists():
    PACKET = LOCAL_PACKET.resolve()
else:
    PACKET = Path("morocco_data_packet").resolve()
    for folder in ("exports", "reference"):
        if not (PACKET / folder).exists():
            print(f"Downloading {folder}.zip from the course data release ...")
            zip_file, _ = urllib.request.urlretrieve(f"{DATA_RELEASE}/{folder}.zip")
            with zipfile.ZipFile(zip_file) as zf:
                zf.extractall(PACKET)

# Fail LOUD on anything missing, naming the exact file. Today that includes the
# two product-space layout files that ship inside reference/.
required = [
    PACKET / "exports" / "outputs.parquet",
    PACKET / "exports" / "densities.parquet",
    PACKET / "reference" / "fields.csv",
    PACKET / "reference" / "units.csv",
    PACKET / "reference" / "umap_layout_hs92.csv",
    PACKET / "reference" / "top_edges_hs92.csv",
]
for f in required:
    if not f.exists():
        raise FileNotFoundError(
            f"Required file not found: {f}\n"
            "If you downloaded the packet before Day 2, delete your local "
            "morocco_data_packet folder and re-run this cell so the updated "
            f"reference.zip (with the product-space layout) is fetched from {DATA_RELEASE}."
        )

print(f"Packet found: {PACKET}")
print(f"Focus country: {COUNTRY} | comparators: {COMPARATORS} | year: {YEAR}")

In [ ]:
# Load the tables for today. outputs.parquet and fields.csv are Day 1's tables;
# the two layout files position the product space exactly as the Atlas draws it.
exports = pd.read_parquet(PACKET / "exports" / "outputs.parquet")
fields = pd.read_csv(PACKET / "reference" / "fields.csv")
units = pd.read_csv(PACKET / "reference" / "units.csv")

# The packet stores measure columns as float32 to keep the download small.
# Upcast before computing: the self-checks below assert identities to 1e-9.
float32_cols = exports.select_dtypes("float32").columns
exports[float32_cols] = exports[float32_cols].astype("float64")

prod_name = fields.set_index("Field ID")["Field Name"]
print("exports rows:", f"{len(exports):,}")

## 1. Rebuild yesterday's objects (from memory, then check)

Everything today grows out of the Mcp matrix you built on Day 1, so we rebuild it in one screen: shares, RCA, binarize, count. If any line surprises you, that is useful information about what to review.

In [ ]:
# Country x product matrix of export values for YEAR, then shares (Day 1, section 2).
year_long = exports.query("Period == @YEAR")[["Unit", "Field ID", "Outputs"]]
X = year_long.pivot_table(index="Unit", columns="Field ID", values="Outputs", fill_value=0.0)
shares = X.div(X.sum(axis=1), axis=0)

# COMPLETION exercise: yesterday's key line, from memory. Fill in ONLY the
# world-share denominator; the RCA division is already done for you.
#
# PROMPT: `world_share` is each product's share of TOTAL world exports: the column
# sum of X divided by the grand total of X.
# TODO(core): complete the one missing line below.
rca = shares.div(world_share, axis=1)  # worked for you: share ÷ world share

# Binarize and count (Day 1, sections 4 and 5), worked today because you own them.
mcp = (rca > 1).astype(float)
diversity = mcp.sum(axis=1)
ubiquity = mcp.sum(axis=0)

print(f"{COUNTRY} diversity:", int(diversity.loc[COUNTRY]))
print(f"matrix: {X.shape[0]} countries x {X.shape[1]} products")

In [ ]:
# Self-check: the Day 1 handshake. Shares are shares, RCA satisfies the Balassa
# identity, and Morocco's diversity is exactly yesterday's 129 (same data, same
# method, same number; if it moved, a line above changed meaning).
assert (shares.sum(axis=1) - 1).abs().max() < 1e-9, (
    "shares must sum to 1 per country. Divide each row by its own row total."
)
_ww = X.sum(axis=1) / X.sum().sum()
assert (rca.mul(_ww, axis=0).sum(axis=0).dropna() - 1).abs().max() < 1e-6, (
    "the world-export-weighted mean of RCA must be 1 for every product (Balassa "
    "identity). Check the world_share completion: column sums over the grand total."
)
if COUNTRY == "MA" and YEAR == 2023:
    assert int(diversity.loc[COUNTRY]) == 129, (
        "Morocco's 2023 diversity was 129 on Day 1 and must be 129 today. "
        "If it differs, the RCA or binarization step above drifted."
    )
print("OK: yesterday's pipeline reproduced.")

## 2. Closing yesterday's loose end: complexity without the threshold

On Day 1 you saw that moving the RCA cutoff from 1 to 0.5 or 2 changes what counts as "present", and the choice felt arbitrary. This morning's first lecture gave the answer: do not threshold at all. Transform RCA into a bounded weight and run the same algebra on it:

$$ \widehat{M}_{cp} = \frac{\text{RCA}_{cp}}{1 + \text{RCA}_{cp}} \in [0, 1) $$

An RCA of 1 maps to 0.5, an RCA of 9 maps to 0.9, and nothing is ever thrown away. The lecture's claim: rankings get more stable and the index tracks income better. Let us see what changes in our data.

### Predict-before-run #1

Commit an answer before running. If we drop the threshold entirely and use the continuous weights, does the country ranking change a little or a lot? And which kind of country moves most: diversified manufacturers, or concentrated resource exporters?

In [ ]:
# TODO(core): build the continuous presence matrix `m_hat` from `rca`.
#
# PROMPT: apply the Hausmann transformation cell by cell: rca / (1 + rca).
# DataFrames divide elementwise, so this is one line with no loop.
# TODO(core): write your code here
raise NotImplementedError()

print("RCA -> M-hat examples: 0.5 ->", round(0.5 / 1.5, 3), "| 1 -> 0.5 | 9 -> 0.9")
print(f"{COUNTRY} M-hat for Passenger Cars:", round(m_hat.loc[COUNTRY, "P - 8703"], 3))

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the formula. Each cell of `m_hat` is that cell's RCA divided by one plus that cell's RCA. No sums, no axes.

Hint 2, pandas. Arithmetic on a DataFrame is elementwise, so `rca / (1 + rca)` computes the transformation for every country and product at once.

Still stuck? Then `m_hat = rca / (1 + rca)`. Check one value: a cell with RCA exactly 1 must become 0.5.
</details>

In [ ]:
# The Day 1 eigenvector machinery, packaged as a function you read rather than
# retype. It is EXACTLY Day 1 section 6: normalize both ways, form the
# product-product reflection matrix, take the SECOND eigenvector, sign-fix to
# diversity, standardize. Lecture 3's point is that the same formulas run
# unchanged on the continuous M-hat: generalized "diversity" is a row sum of
# weights, generalized "ubiquity" a column sum, and nothing else moves.
def reflections_index(m: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """ECI and PCI as the second eigenvector of the reflections operator.

    Accepts the binary Mcp or any non-negative weight matrix (e.g. M-hat).
    Returns (eci, pci), each standardized to mean 0 and sd 1.
    """
    m = m.loc[m.sum(axis=1) > 0, m.sum(axis=0) > 0]  # drop empty rows/columns
    row_norm = m.div(m.sum(axis=1), axis=0)  # each row sums to 1
    col_norm = m.div(m.sum(axis=0), axis=1)  # each column sums to 1
    Mpp = col_norm.values.T @ row_norm.values  # product-product reflections
    eigvals, eigvecs = np.linalg.eig(Mpp)
    order = np.argsort(-eigvals.real)  # eigenvalues, largest first
    kp = eigvecs[:, order[1]].real  # the SECOND eigenvector (1st is trivial)
    kc = row_norm.values @ kp  # project products back onto countries
    sign = 1 if np.corrcoef(m.sum(axis=1).values, kc)[0, 1] > 0 else -1
    eci = pd.Series(sign * kc, index=m.index)
    pci = pd.Series(sign * kp, index=m.columns)
    return (eci - eci.mean()) / eci.std(), (pci - pci.mean()) / pci.std()


eci_bin, pci_bin = reflections_index(mcp)  # yesterday's index, recomputed
eci_con, _ = reflections_index(m_hat)  # the no-threshold variant

common_c = eci_bin.index.intersection(eci_con.index)
rho_bc = spearmanr(eci_bin[common_c], eci_con[common_c]).statistic
print(f"Spearman(binary ECI, continuous ECI) = {rho_bc:.3f} over {len(common_c)} countries")
print(
    f"{COUNTRY}: binary ECI {eci_bin[COUNTRY]:.2f} (rank {int(eci_bin.rank(ascending=False)[COUNTRY])}) "
    f"| continuous ECI {eci_con[COUNTRY]:.2f} (rank {int(eci_con.rank(ascending=False)[COUNTRY])})"
)

# Who moves most when the threshold goes away? Signed rank shift, so you can
# read the direction: negative = the country RISES once the threshold is gone.
rank_shift = eci_con[common_c].rank(ascending=False) - eci_bin[common_c].rank(ascending=False)
movers = rank_shift.reindex(rank_shift.abs().nlargest(6).index).astype(int)
print("Biggest movers (rank places; negative = rises without the threshold):")
print(movers.to_string())

In [ ]:
# Binary vs continuous ECI, one point per country, the focus country flagged.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(eci_bin[common_c], eci_con[common_c], s=12, alpha=0.4)
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, color="grey", lw=1, ls="--", label="no change")
for code in [COUNTRY, *COMPARATORS]:
    if code in common_c:
        ax.scatter(
            eci_bin[code],
            eci_con[code],
            s=90,
            color="crimson" if code == COUNTRY else "darkorange",
            zorder=5,
        )
        ax.annotate(
            code,
            (eci_bin[code], eci_con[code]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=10,
        )
ax.set_xlabel("ECI from binary Mcp (RCA > 1)")
ax.set_ylabel("ECI from continuous M-hat (no threshold)")
ax.set_title(f"Same algebra, no threshold ({YEAR}): rankings mostly survive")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the transformation and the comparison, as invariants.
#   (1) M-hat lives in [0, 1) and maps RCA = 1 to exactly 0.5 (the definition).
#   (2) M-hat preserves order: wherever RCA is larger, M-hat is larger.
#   (3) The two rankings agree strongly but NOT perfectly. Perfect agreement
#       would mean the threshold never mattered; weak agreement would mean the
#       index is fragile. The lecture's claim lives in between.
assert float(m_hat.values.min()) >= 0 and float(m_hat.values.max()) < 1, (
    "M-hat must lie in [0, 1). It is rca / (1 + rca), not rca / rca.max()."
)
_probe = rca.iloc[:5, :5].values
assert np.allclose(_probe / (1 + _probe), m_hat.iloc[:5, :5].values), (
    "M-hat must equal rca / (1 + rca) cell for cell. Apply the Hausmann "
    "transformation to the WHOLE rca DataFrame."
)
assert 0.85 < rho_bc < 0.9999, (
    f"binary and continuous ECI should rank countries almost, but not exactly, "
    f"alike (got Spearman {rho_bc:.3f}). Far lower means an eigenvector or sign "
    "error; exactly 1.0 means you compared an index to itself."
)
if COUNTRY == "MA" and YEAR == 2023:
    assert abs(eci_bin[COUNTRY] - (-0.47)) < 0.05, (
        "the binary ECI should reproduce Day 1's Morocco value of about -0.47."
    )
print("OK: M-hat is the Hausmann transformation and the rankings mostly survive.")

### Interpret (no AI)

Take the top country in your own movers table. Find it in the scatter (or add it to `COMPARATORS` and re-run the cell) and note which side of the dashed line it sits on, then check what kind of economy it is. The biggest movers tend to be small or resource-concentrated economies, the same pattern the lecture showed for Saudi Arabia, the UAE, and Kuwait; Morocco's own continuous ECI printed above its binary one here. Using what M-hat does to an RCA of 0.8 versus an RCA of 0.05, explain why YOUR top mover moved in the direction it did, and what that says about reading any single ECI number without knowing the presence rule behind it.

TODO(concept): your answer here.

From here on we use the binary Mcp, as the product-space tradition and the packages do. You now know the robustness check to run when a threshold decision worries you.

## 3. Proximity, first by hand on the toy table

The morning defined proximity between two products as the minimum of the two conditional probabilities of co-export:

$$ \phi_{pp'} = \min\{\, P(M_{cp}=1 \mid M_{cp'}=1),\; P(M_{cp'}=1 \mid M_{cp}=1) \,\} $$

Each conditional probability is just counting: of the countries that export one product competitively, what fraction also export the other? We compute it once by hand on Day 1's toy table, where you can count the countries on your fingers.

In [ ]:
# Day 1's toy table (the morning lecture's 3 countries x 4 products), compressed
# to its Mcp. Rebuilt from the raw values so nothing is asserted from memory.
# fmt: off
toy = pd.DataFrame(
    [[ 10,  40, 100, 300],   # Country A
     [150, 100,  10,   0],   # Country B
     [ 20, 200,   1,   0]],  # Country C
    index=["Country A", "Country B", "Country C"],
    columns=["Wheat", "Textiles", "Motorcycles", "Cell phones"],
).astype(float)
# fmt: on
toy_shares = toy.div(toy.sum(axis=1), axis=0)
toy_rca = toy_shares.div(toy.sum(axis=0) / toy.sum().sum(), axis=1)
toy_mcp = (toy_rca > 1).astype(int)
toy_ubiquity = toy_mcp.sum(axis=0)
print("toy Mcp:")
print(toy_mcp)
print("\ntoy ubiquity:", toy_ubiquity.to_dict())

In [ ]:
# TODO(core): the proximity between Wheat and Textiles, by counting.
#
# PROMPT: first count the countries whose Mcp is 1 for BOTH Wheat and Textiles
# (multiply the two columns and sum). Then divide that co-export count by the
# LARGER of the two ubiquities. Dividing by the larger ubiquity IS the min of
# the two conditional probabilities: same numerator, bigger denominator.
# TODO(core): write your code here
raise NotImplementedError()

print("countries exporting both Wheat and Textiles:", toy_both)
print("phi(Wheat, Textiles) =", toy_phi_wt)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the numerator. A country co-exports the pair when both its Mcp entries are 1, so the product of the two columns is 1 exactly for co-exporters. Sum it.

Hint 2, the denominator. P(Wheat given Textiles) divides by Textiles' ubiquity, P(Textiles given Wheat) divides by Wheat's ubiquity. The smaller probability is the one with the larger denominator, so take `max(ubiquity_wheat, ubiquity_textiles)`.

Still stuck? Then `toy_both = int((toy_mcp["Wheat"] * toy_mcp["Textiles"]).sum())` and `toy_phi_wt = toy_both / max(toy_ubiquity["Wheat"], toy_ubiquity["Textiles"])`. Read the two conditional probabilities off the toy table to see which one the min picked.
</details>

In [ ]:
# Self-check: exact by construction on the toy. Only Country B exports both
# Wheat and Textiles, Wheat's ubiquity is 1 and Textiles' is 2, so
# phi = 1 / max(1, 2) = 0.5.
assert toy_both == 1, (
    "exactly ONE toy country (Country B) exports both Wheat and Textiles "
    "competitively. Multiply the two Mcp columns and sum the result."
)
assert abs(toy_phi_wt - 0.5) < 1e-12, (
    "phi(Wheat, Textiles) must be 0.5: one co-exporter over max(ubiquity) = 2. "
    "If you got 1.0 you divided by the SMALLER ubiquity, which is the max of "
    "the conditionals, not the min."
)
print("OK: phi(Wheat, Textiles) = 0.5, counted by hand.")

## 4. Proximity on the real matrix

Now the same counting for all 862 products at once. The co-export counts for every pair come from one matrix product, and the pair counts sit on top of a familiar object: the diagonal of that matrix is exactly yesterday's ubiquity.

In [ ]:
# TODO(core): the co-export count matrix. Entry (p, p') = number of countries
# with Mcp = 1 for BOTH p and p'.
#
# PROMPT: this is the matrix product of Mcp transposed with Mcp: `mcp.T @ mcp`.
# Row p of mcp.T meets column p' of mcp exactly at the countries exporting both.
# TODO(core): write your code here
raise NotImplementedError()

print("co-export matrix shape:", co_export.shape)
print(
    "countries co-exporting Passenger Cars and Insulated Wire (P - 8544):",
    int(co_export.loc["P - 8703", "P - 8544"]),
)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the shape. You want one number per PAIR of products, an 862 by 862 matrix, from a countries-by-products Mcp. That calls for a transpose on one side.

Hint 2, the product. `mcp.T @ mcp` multiplies (products x countries) by (countries x products): each cell sums, over countries, the product of two 0/1 entries, which is 1 only when the country exports both.

Still stuck? Then `co_export = mcp.T @ mcp`. Check its diagonal against yesterday's ubiquity before moving on; the self-check below does exactly that.
</details>

In [ ]:
# Self-check: an identity ties your matrix to yesterday's object. The diagonal
# of the co-export matrix is each product's co-export count with ITSELF, which
# is just the number of countries exporting it: the ubiquity. A pasted or
# hand-built matrix will not satisfy this cell for cell.
assert co_export.shape == (mcp.shape[1], mcp.shape[1]), (
    "co_export must be product x product. If it came out country x country you "
    "multiplied mcp @ mcp.T instead of mcp.T @ mcp."
)
assert np.allclose(np.diag(co_export.values), ubiquity.values), (
    "the diagonal of the co-export matrix must equal ubiquity exactly: a product "
    "is co-exported with itself by every country that exports it."
)
assert np.allclose(co_export.values, co_export.values.T), (
    "co-export counts are symmetric by construction; check the matrix product."
)
print("OK: co-export counts verified against yesterday's ubiquity.")

In [ ]:
# From counts to proximity, worked for you. Dividing the co-export count by each
# product's ubiquity gives the two conditional probabilities; the elementwise
# max of the two ubiquities in the denominator implements the min of the two
# probabilities in one step. The diagonal is set to 0: a product is not related
# to itself for our purposes, and the density denominator below must not count it.
denom = np.maximum.outer(ubiquity.values, ubiquity.values)
with np.errstate(divide="ignore", invalid="ignore"):
    phi_values = np.where(denom > 0, co_export.values / denom, 0.0)
np.fill_diagonal(phi_values, 0.0)
phi = pd.DataFrame(phi_values, index=mcp.columns, columns=mcp.columns)

print(f"phi: {phi.shape[0]} x {phi.shape[1]}, max {phi.values.max():.3f}")
print("phi(Women's Knit Shirts, Women's Knit Suits) =", round(phi.loc["P - 6106", "P - 6104"], 3))

The morning's flagship pair lands where the lecture said it would: women's shirts and women's suits sit near the very top of all pairs. The lecture's table showed 0.967 on 2010 data; on our single year of 2023 data the value differs but the pair is still in the top tenth of a percent. Method stable, number not identical, which is the usual shape of real replication.

In [ ]:
# Why the min? The lecture's wine and ostrich eggs, found in OUR data. Take the
# most asymmetric pair: Oxometallic Salts (yesterday's top-PCI product, made by
# 2 countries) and Non-Aqueous Paints (made by 42).
p_rare, p_common = "P - 2841", "P - 3208"
both = int(co_export.loc[p_rare, p_common])
p_common_given_rare = both / ubiquity[p_rare]
p_rare_given_common = both / ubiquity[p_common]
print(f"P({prod_name[p_common]} | {prod_name[p_rare]}) = {p_common_given_rare:.2f}")
print(f"P({prod_name[p_rare]} | {prod_name[p_common]}) = {p_rare_given_common:.2f}")
print(f"min (our phi) = {phi.loc[p_rare, p_common]:.2f}")

Every exporter of Oxometallic Salts also exports paints, so one conditional probability is 1.00. But almost no paint exporter makes Oxometallic Salts. Taking the max would declare the pair strongly related on the strength of two countries; the min keeps only relationships confirmed in both directions. That is the same reason exporting ostrich eggs should not make wine look nearby.

In [ ]:
# Self-check: structural properties of a proximity matrix, plus the toy and the
# lecture's pair. None of these can be satisfied by a constant or pasted matrix.
assert np.allclose(phi.values, phi.values.T), (
    "phi must be symmetric (it is a min of a symmetric pair)."
)
assert float(phi.values.min()) >= 0 and float(phi.values.max()) <= 1, (
    "phi is a probability, so every entry must lie in [0, 1]."
)
assert np.allclose(np.diag(phi.values), 0), (
    "the diagonal must be 0; a product's proximity to itself is excluded so the "
    "density denominator below does not count it."
)
# The same recipe applied to the toy table must reproduce the hand count.
_td = np.maximum.outer(toy_ubiquity.values, toy_ubiquity.values)
_tphi = np.where(_td > 0, (toy_mcp.T @ toy_mcp).values / _td, 0.0)
np.fill_diagonal(_tphi, 0.0)
assert abs(_tphi[0, 1] - 0.5) < 1e-12 and abs(_tphi[2, 3] - 1.0) < 1e-12, (
    "on the toy table the recipe must give phi(Wheat, Textiles) = 0.5 and "
    "phi(Motorcycles, Cell phones) = 1.0. If not, the denominator is not the "
    "elementwise max of the two ubiquities."
)
# The lecture's flagship pair sits in the top 0.1% of all product pairs.
_upper = phi.values[np.triu_indices_from(phi.values, k=1)]
_pair_pct = (_upper < phi.loc["P - 6106", "P - 6104"]).mean()
assert _pair_pct > 0.99, (
    "women's knit shirts (P - 6106) and suits (P - 6104) must land near the top "
    f"of all pairs (got percentile {_pair_pct:.3f}). If they are mid-ranked, the "
    "numerator and denominator are misaligned."
)
print(
    "OK: phi is symmetric, bounded, zero-diagonal, matches the toy, and the "
    "shirts/suits pair is top-0.1%."
)

In [ ]:
# Cross-check against the maintained package. Day 1's lesson continues: you have
# now hand-rolled proximity once, so from tomorrow you call ecomplexity and
# trust it, because you have seen it agree with your own code.
from ecomplexity import proximity  # noqa: E402  (import next to its first use)

pkg_input = year_long.rename(columns={"Unit": "loc", "Field ID": "prod", "Outputs": "val"}).assign(
    time=YEAR
)
PKG_COLS = {"time": "time", "loc": "loc", "prod": "prod", "val": "val"}
phi_pkg = proximity(pkg_input, PKG_COLS)
phi_pkg_series = phi_pkg.set_index(["prod_1", "prod_2"])["proximity"]

# Align a sample of pairs (the full matrix has 743k pairs; 20k is plenty).
# Drop the diagonal first: we zeroed it by convention, the package does not.
_ours = phi.stack()
_ours = _ours[_ours.index.get_level_values(0) != _ours.index.get_level_values(1)]
_common = _ours.sample(20_000, random_state=0).index.intersection(phi_pkg_series.index)
rho_pkg = spearmanr(_ours[_common], phi_pkg_series[_common]).statistic
print(f"Spearman(our phi, ecomplexity phi) = {rho_pkg:.4f} over {len(_common):,} sampled pairs")

In [ ]:
# Self-check: the package agrees with the hand-roll (same definition, same data).
assert rho_pkg > 0.99, (
    f"our phi and the ecomplexity package's phi should rank pairs identically "
    f"(got Spearman {rho_pkg:.3f}). A gap here means the hand-rolled formula "
    "deviates from min-of-conditionals."
)
print("OK: the package reproduces the hand-rolled proximity.")

## 5. The product space: place your country on the map

The morning showed you the product space twice: the 2007 Science version and the Atlas's current one. We now draw the current one for your country, using the Growth Lab's published layout. The node positions come from UMAP, the dimensionality-reduction method the networks lecture closed with, and the edge list keeps each product's strongest links, so the file you are loading is the output of the exact visualization toolkit from this morning: prune the dense matrix, then lay out what survives.

### Predict-before-run #2

Commit an answer. When the product space appears, which product family sits at its dense, tightly connected core: oil and minerals, agriculture, or machinery and chemicals? And will your country's highlighted products sit mostly in that core or around the periphery?

In [ ]:
from IPython.display import HTML  # noqa: E402  (kept next to its one use)

# Load the layout: one row per product (position + cluster), one row per edge.
layout_nodes = pd.read_csv(
    PACKET / "reference" / "umap_layout_hs92.csv", dtype={"product_hs92_code": str}
)
layout_edges = pd.read_csv(PACKET / "reference" / "top_edges_hs92.csv", dtype=str)

# HS92 codes need zero-padding to 4 digits to join our Field IDs (0714, not 714).
layout_nodes["Field ID"] = "P - " + layout_nodes["product_hs92_code"].str.zfill(4)
nodes = (
    layout_nodes.merge(fields[["Field ID", "Field Name"]], on="Field ID", how="left")
    .assign(
        world_exports=lambda d: d["Field ID"].map(X.sum(axis=0)),
        in_basket=lambda d: d["Field ID"].map(mcp.loc[COUNTRY]).fillna(0).astype(bool),
    )
    .dropna(subset=["world_exports"])
)

# Colors follow the Atlas cluster families (the legend the morning slide used).
CLUSTER_COLORS = {
    "Agriculture": "#F5C548",
    "Minerals": "#7A6552",
    "Industrial Chemicals and Metals": "#A05FB5",
    "Metalworking and Electrical Machinery and Parts": "#D7484B",
    "Electronic and Electrical Goods": "#4FB8C4",
    "Construction, Building, and Home Supplies": "#E08544",
    "Textile and Home Goods": "#8B8B8B",
    "Textile Apparel and Accessories": "#5FA755",
}

# Edges ship mirrored (each pair appears both ways); keep each pair once.
pair_sorted = np.sort(layout_edges.values, axis=1)
pairs = pd.DataFrame(pair_sorted, columns=["a", "b"]).drop_duplicates()
xy = layout_nodes.set_index("product_hs92_code")[["product_space_x", "product_space_y"]]
edge_x, edge_y = [], []
for a, b in pairs.values:
    if a in xy.index and b in xy.index:
        edge_x += [xy.loc[a, "product_space_x"], xy.loc[b, "product_space_x"], None]
        edge_y += [xy.loc[a, "product_space_y"], xy.loc[b, "product_space_y"], None]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(color="#DDDDDD", width=0.6),
        hoverinfo="skip",
        showlegend=False,
    )
)
# Node size scales with the square root of world trade, as on the Atlas.
size = 3 + 15 * np.sqrt(nodes["world_exports"] / nodes["world_exports"].max())
for cluster, color in CLUSTER_COLORS.items():
    grp = nodes["product_space_cluster_name"] == cluster
    for basket, opacity, outline in [(False, 0.25, 0), (True, 1.0, 1.2)]:
        sel = grp & (nodes["in_basket"] == basket)
        if not sel.any():
            continue
        fig.add_trace(
            go.Scatter(
                x=nodes.loc[sel, "product_space_x"],
                y=nodes.loc[sel, "product_space_y"],
                mode="markers",
                name=cluster,
                legendgroup=cluster,
                showlegend=(not basket),  # one legend entry per cluster
                marker=dict(
                    size=size[sel],
                    color=color,
                    opacity=opacity,
                    line=dict(width=outline, color="black"),
                ),
                text=nodes.loc[sel, "Field Name"] + " (" + nodes.loc[sel, "Field ID"] + ")",
                hoverinfo="text",
            )
        )
fig.update_layout(
    title=f"The product space, {YEAR}. Highlighted: products {COUNTRY} exports competitively.",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    height=650,
    legend=dict(font=dict(size=10)),
)
# Embedded inline so the one interactive figure survives offline export (the
# same reasoning as Day 1's treemap; see that notebook's note).
HTML(fig.to_html(full_html=False, include_plotlyjs="inline"))

Read the picture before scrolling on. The dense core is machinery, chemicals, and metalworking, not oil or agriculture. Highlighted products in the core have many strong neighbours; highlighted products on the periphery connect to little. Where a country's basket sits on this map is a summary of what it can reach next, which is exactly what we quantify in the next section.

In [ ]:
# Self-check: the join actually worked and the highlight is the basket.
_matched = nodes["Field ID"].isin(X.columns).mean()
assert _matched > 0.95, (
    f"only {_matched:.0%} of layout products matched the trade matrix. The HS92 "
    "codes must be zero-padded to 4 digits before prefixing with 'P - '."
)
_expected = int(mcp.loc[COUNTRY][mcp.columns.isin(nodes["Field ID"])].sum())
assert int(nodes["in_basket"].sum()) == _expected, (
    "the number of highlighted nodes must equal the country's diversity "
    "restricted to products present in the layout. Check the in_basket mapping."
)
print(
    f"OK: layout joined ({_matched:.1%} matched) and {int(nodes['in_basket'].sum())} "
    f"products highlighted for {COUNTRY}."
)

## 6. Density: how close is each product to what you already do?

The morning's closing slide defined relatedness density in one line: the share of a product's relatedness that your country already has. For country c and product p, sum the proximities from p to the products c already exports, and divide by the sum of p's proximities to everything:

$$ \omega_{cp} = \frac{\sum_{p'} M_{cp'}\,\phi_{p'p}}{\sum_{p'} \phi_{p'p}} $$

A density of 0.25 means a quarter of the product's total relatedness is already inside your basket. This is the single number the Principle of Relatedness runs on, and tomorrow it becomes the regressor in the entry regressions.

### Fix the broken cell: the missing denominator, again

The worked line below stops at the numerator: a raw sum of proximities. Print `density_numerator.max().max()` and watch it sail far past 1, which no share of anything can do. Products with many neighbours would score high for every country alike. Day 1's bug compared raw values instead of shares; this is the same disease one level up. Your job is the line marked `# TODO(core)`: normalize each product's sum by its total relatedness, then run the self-check.

In [ ]:
# Goal: density for every country x product pair, bounded in [0, 1].
# The matrix product below is the NUMERATOR: for each country row, it adds up
# the proximities from each product to everything the country already exports.
density_numerator = mcp @ phi

# TODO(core): the line below is the bug: it stops at the numerator. Replace it so
# the numerator is divided by each product's TOTAL proximity to all products
# (the column sums of phi), making density the SHARE of a product's relatedness
# the country already covers.
#
# PROMPT: divide density_numerator by phi.sum(axis=0), matching on the product
# axis: .div(phi.sum(axis=0), axis=1).
# TODO(core): write your code here
raise NotImplementedError()

print(
    f"{COUNTRY} density: min {density.loc[COUNTRY].min():.3f}, "
    f"max {density.loc[COUNTRY].max():.3f}"
)
# Diagnosis (say it aloud): without the denominator, "close to everything" just
# means "the product has many neighbours", the same for every country. Only the
# normalized share says how close the product is to YOUR basket.

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the disease. Print `density_numerator.max().max()`: values far above 1 cannot be a share of anything. The numerator needs a denominator.

Hint 2, the denominator. Each product p's denominator is the same for every country: p's total proximity to all products, which is a column sum of phi, `phi.sum(axis=0)`. Divide the numerator by it along the product axis.

Still stuck? Then `density = density_numerator.div(phi.sum(axis=0), axis=1)`. Confirm the maximum is now below 1.
</details>

In [ ]:
# Self-check, three layers, echoing Day 1's two-sided bug check.
#   (1) The fix is in: density equals numerator over column sums, cell for cell,
#       and is NOT still the raw numerator.
#   (2) Bounds: a share lives in [0, 1].
#   (3) The invariant a shortcut cannot fake: for every country, the products it
#       ALREADY exports sit at higher average density than the ones it does not.
#       Relatedness concentrates around what you already do; a constant matrix,
#       or a transposed formula, breaks this.
_ref = (mcp @ phi).div(phi.sum(axis=0), axis=1)
assert np.allclose(density.values, _ref.values, equal_nan=True), (
    "density must be (mcp @ phi) divided by phi's COLUMN sums, matched on the "
    "product axis (.div(..., axis=1))."
)
assert not np.allclose(density.values, density_numerator.values), (
    "density is still the raw numerator. Divide by phi.sum(axis=0) so it becomes "
    "a share of each product's total relatedness."
)
assert float(density.values.min()) >= 0 and float(density.values.max()) <= 1 + 1e-9, (
    "density is a share and must lie in [0, 1]. Values above 1 mean the "
    "denominator is missing or on the wrong axis."
)
_in = (density * mcp).sum(axis=1) / mcp.sum(axis=1)
_out = (density * (1 - mcp)).sum(axis=1) / (1 - mcp).sum(axis=1)
_valid = (mcp.sum(axis=1) > 0) & ((1 - mcp).sum(axis=1) > 0)
assert (_in[_valid] > _out[_valid]).mean() > 0.95, (
    "for nearly every country, average density must be higher on products it "
    "already exports than on products it does not. If this fails, the matrix "
    "orientation (countries x products) got flipped somewhere."
)
print(
    f"OK: density is a bounded share, and {COUNTRY}'s basket sits at density "
    f"{_in[COUNTRY]:.3f} vs {_out[COUNTRY]:.3f} outside it."
)

In [ ]:
# One call to the maintained package now reproduces the whole week so far: RCA,
# Mcp, diversity, ubiquity, ECI, PCI, and today's density, plus the two outlook
# measures we use in the next section. (check_logsupermodularity=False silences
# an optional diagnostic related to this morning's theory lecture.)
from ecomplexity import ecomplexity  # noqa: E402

cdata = ecomplexity(pkg_input, PKG_COLS, check_logsupermodularity=False)
mine = density.loc[COUNTRY]
pkg = cdata.query("loc == @COUNTRY").set_index("prod")["density"]
_common_p = mine.index.intersection(pkg.index)
rho_dens = spearmanr(mine[_common_p], pkg[_common_p]).statistic
print(f"Spearman(our density, ecomplexity density) = {rho_dens:.4f}")

In [ ]:
# Self-check: the package agrees with the hand-roll to rank precision. (The last
# decimals can differ from small implementation choices; the ranking must not.)
assert rho_dens > 0.99, (
    f"our density and the package's should rank products nearly identically "
    f"(got Spearman {rho_dens:.3f}). Check the numerator orientation and the "
    "column-sum denominator."
)
print("OK: the package reproduces the hand-rolled density.")

In [ ]:
# Validation against WIPO's published density, as on Day 1. Expect agreement to
# be REAL but noticeably weaker than yesterday's PCI comparison.
wipo_density = (
    pd.read_parquet(PACKET / "exports" / "densities.parquet")
    .query("Period == @YEAR and Unit == @COUNTRY")
    .set_index("Field ID")["Relatedness Density"]
)
_common_w = mine.index.intersection(wipo_density.index)
rho_wipo = spearmanr(mine[_common_w], wipo_density[_common_w]).statistic
print(f"Spearman(our density, WIPO density) = {rho_wipo:.3f} over {len(_common_w)} products")

In [ ]:
# Self-check: positive, real, and imperfect, by design. Density inherits every
# upstream choice twice over: WIPO's presence rule differs (Day 1's lesson) AND
# its proximity matrix differs, and the two differences compound.
assert 0.2 < rho_wipo < 0.95, (
    f"our density should agree with WIPO's directionally but imperfectly "
    f"(got {rho_wipo:.3f}). Near 0 suggests a broken pipeline; near 1 would "
    "mean you loaded WIPO's column instead of computing your own."
)
print(f"OK: real but imperfect agreement with WIPO ({rho_wipo:.3f}), as expected.")

### Interpret (no AI)

Yesterday your PCI agreed with WIPO's at a Spearman around 0.92. Today your density agrees at the weaker value printed above. Both gaps come from methodology, not bugs. Explain why a DERIVED measure like density shows a wider gap than the index it is built from: name the two upstream choices it inherits, and say what that implies about comparing density numbers across two reports that do not document their presence rule and proximity matrix.

TODO(concept): your answer here.

## 7. The payoff: a ranked shortlist of diversification opportunities

Cross the two numbers you now own. Density says how reachable a product is from your basket; PCI (yesterday) says how sophisticated it is. The interesting corner is high density and high PCI: nearby and worth having. The package's two outlook measures summarize the same trade-off: COI (complexity outlook) adds up how much sophistication sits near your basket overall, and COG (complexity outlook gain) says how much new reach a specific product would unlock if you entered it.

### Predict-before-run #3

Before the table appears, write down ONE product you believe is realistically "nearby" for your country, something adjacent to what it already does. Then find it (or its family) in the ranked table and note its density rank. Most people's intuitions run more high-tech than the data.

In [ ]:
# Candidates: products the country does NOT yet export competitively.
ma_candidates = (
    pd.DataFrame(
        {
            "density": density.loc[COUNTRY],
            "pci": pci_bin,
            "world_exports": X.sum(axis=0),
            "rca": rca.loc[COUNTRY],
        }
    )
    .query("rca < 1")
    .dropna(subset=["pci"])
)

# Attach names and the package's COG (how much outlook entering each product adds).
cog = cdata.query("loc == @COUNTRY").set_index("prod")["cog"]
ma_candidates["cog"] = cog
ma_candidates["product"] = prod_name

coi_value = float(cdata.query("loc == @COUNTRY")["coi"].iloc[0])
print(
    f"{COUNTRY} complexity outlook (COI): {coi_value:.2f} "
    "(one number for the whole basket's frontier)"
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    ma_candidates["density"],
    ma_candidates["pci"],
    s=3 + 120 * np.sqrt(ma_candidates["world_exports"] / ma_candidates["world_exports"].max()),
    alpha=0.35,
)
pci_median = ma_candidates["pci"].median()
ax.axvline(ma_candidates["density"].median(), color="grey", lw=0.8, ls=":")
ax.axhline(pci_median, color="grey", lw=0.8, ls=":")
# Annotate the frontier corner: the top-density products above median PCI.
# Offsets alternate above/below so nearby labels do not sit on one another.
_frontier = ma_candidates.query("pci > @pci_median").nlargest(5, "density")
_offsets = [(6, 8), (6, -14), (6, 22), (6, -28), (6, 36)]
for (fid, row), (dx, dy) in zip(_frontier.iterrows(), _offsets):
    ax.scatter(row["density"], row["pci"], s=60, color="crimson", zorder=5)
    ax.annotate(
        str(row["product"])[:24],
        (row["density"], row["pci"]),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=8,
        arrowprops=dict(arrowstyle="-", lw=0.5, color="grey"),
    )
ax.set_xlabel(f"Density around {COUNTRY}'s basket (share of relatedness already held)")
ax.set_ylabel("PCI (trade-only, from Day 1's method)")
ax.set_title(
    f"Diversification frontier for {COUNTRY}, {YEAR}: nearby (right) and sophisticated (up)"
)
plt.tight_layout()
plt.show()

In [ ]:
# The ranked opportunity table: unexploited products above median PCI, ranked by
# density. This table plus the product space is your Day 5 deck material.
opportunities = (
    ma_candidates.query("pci > @pci_median")
    .nlargest(15, "density")
    .loc[:, ["product", "density", "pci", "cog", "world_exports"]]
    .assign(world_exports=lambda d: (d["world_exports"] / 1e9).round(1))
    .rename(columns={"world_exports": "world_exports_bnUSD"})
    .round({"density": 3, "pci": 2, "cog": 2})
)
opportunities

In [ ]:
# Self-check: the table is what it claims to be.
assert (ma_candidates["rca"] < 1).all(), (
    "candidates must be products the country does NOT yet export competitively "
    "(RCA < 1). Check the query."
)
assert len(opportunities) > 0 and opportunities["density"].is_monotonic_decreasing, (
    "the opportunity table must be non-empty and sorted by density, largest first."
)
assert opportunities["cog"].notna().all(), (
    "every opportunity row needs its COG from the package output. Check the join on product IDs."
)
print(f"OK: {len(opportunities)} ranked opportunities for {COUNTRY}.")

### Interpret (no AI, this is the AI red-light cell)

The model ranks; you judge. Pick ONE product from your own top-ten table and argue for DROPPING it from a shortlist you would hand to a minister, on grounds the data does not contain: water intensity, land, geopolitics, a competitor's lock-in, environmental cost, or something else you know about your country. Two or three sentences. There is no correct answer, but "the density is high so we keep it" is a wrong one.

TODO(concept): your answer here.

One honesty note before you take this table to your team. Proximity is revealed co-location, not a causal recipe: it says shirt-exporters tend to export suits, not why. Density ranks nearness; whether nearby products are actually ENTERED more often is an empirical claim, and it is exactly what tomorrow's regressions test. And all of it lives on trade data alone: services, informality, and everything Morocco sells domestically are invisible here.

## 8. Recall: close the loop (no scrolling up)

From memory:

1. Proximity is the minimum of two conditional probabilities. Which probabilities, and why the minimum rather than the maximum?
2. Your country's density for a product is 0.30. Say precisely what that number means.
3. What does COG add that density alone does not?
4. Tomorrow we regress entry on density. What sign do we expect, and what result would falsify the Principle of Relatedness?

TODO(concept): your answers here.

## Stretch (for fast finishers, scaffolding removed)

Pick any. No prompts, no self-checks.

(a) Nearest-neighbour countries. The packet's `cross_domain/unit_proximities.parquet` scores how similar two countries' capability profiles are. Download `cross_domain.zip` from the data release if you do not have it, find your country's ten nearest neighbours, and ask whether your project's comparator countries are actually the data's idea of "similar". (This is the producer-space slide from this morning, as a table.)

(b) Other similarity measures. The lecture listed alternatives to min-of-conditionals: Jaccard (co-exports over the union) and cosine similarity between Mcp columns. Compute either for all pairs and rank-correlate it against phi. How much does the choice matter?

(c) A Day 5 teaser. `cross_domain/field_proximities.parquet` holds proximities BETWEEN domains: products, technologies, and science fields in one network. Filter to pairs linking your country's competitive products (`P - `) to technology fields (`T - `), and ask which technologies sit closest to your export strengths. Hold the interpretation for Day 5.

(d) Build your own layout. You already have phi. Prune it (a maximum spanning tree plus all edges above 0.5, `networkx` has both pieces) and compute a spring layout. Compare your picture with the Atlas layout you drew above: which choices did the professionals make differently, and why might they have made them?

In [ ]:
# (Your stretch code here, no scaffold, no self-check.)